In [0]:
%run /Users/sandysakthivel2005@gmail.com/common/03_Logger

In [0]:
print(logger)

<Logger SocialMediaPipeline (WARNING)>


In [0]:
# Databricks notebook source

# MAGIC %run /Users/sandysakthivel2005@gmail.com/common/03_Logger

from pyspark.sql.functions import *

try:

    logger.info("Silver Valid Tweets Pipeline Started")
    print("Silver Valid Tweets Pipeline Started")

    # ==========================================
    # Read Bronze Streaming Table
    # ==========================================

    bronzeDF = (
        spark.readStream
             .table("bronze_catalog1.raw.bronze_valid_tweets1")
    )

    logger.info("Bronze Valid Tweets Table Read Successfully")
    print("Bronze Valid Tweets Table Read Successfully")

    # ==========================================
    # Remove Duplicates
    # ==========================================

    silverDF = bronzeDF.dropDuplicates(["tweet_id"])

    # ==========================================
    # Handle Null Values
    # ==========================================

    silverDF = (
        silverDF
            .fillna({
                "topic_category": "Unknown",
                "impressions": 0,
                "likes": 0,
                "retweets": 0,
                "replies": 0,
                "engagement_count": 0,
                "sentiment_score": 0
            })
    )

    # ==========================================
    # Data Validation
    # ==========================================

    silverDF = (
        silverDF
            .filter(col("tweet_id").isNotNull())
            .filter(col("tweet_text").isNotNull())
            .filter(col("topic_category").isNotNull())
            .filter(col("tweet_timestamp").isNotNull())
            .filter(col("impressions") >= 0)
            .filter(col("likes") >= 0)
            .filter(col("retweets") >= 0)
            .filter(col("replies") >= 0)
            .filter(col("engagement_count") >= 0)
            .filter(col("sentiment_score") >= -1)
            .filter(col("sentiment_score") <= 1)
    )

    # ==========================================
    # Standardize Text
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("tweet_text", trim(col("tweet_text")))
            .withColumn("topic_category", upper(trim(col("topic_category"))))
    )

    # ==========================================
    # Convert Data Types
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("tweet_timestamp", to_timestamp(col("tweet_timestamp")))
            .withColumn("impressions", col("impressions").cast("int"))
            .withColumn("likes", col("likes").cast("int"))
            .withColumn("retweets", col("retweets").cast("int"))
            .withColumn("replies", col("replies").cast("int"))
            .withColumn("engagement_count", col("engagement_count").cast("int"))
            .withColumn("sentiment_score", col("sentiment_score").cast("double"))
    )

    # ==========================================
    # Standardize Date
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("tweet_date", to_date(col("tweet_timestamp")))
    )

    # ==========================================
    # Audit Column
    # ==========================================

    silverDF = (
        silverDF
            .withColumn("silver_load_time", current_timestamp())
            .withColumn("pipeline_name", lit("Silver_Valid_Tweets"))
    )

    logger.info("Silver Valid Tweets Transformations Completed Successfully")
    print("Silver Valid Tweets Transformations Completed Successfully")

    # ==========================================
    # Write Silver Table
    # ==========================================

    silverQuery = (
        silverDF.writeStream
            .trigger(availableNow=True)
            .format("delta")
            .outputMode("append")
            .option(
                "checkpointLocation",
                "abfss://socialmedia@socialmediaadls001.dfs.core.windows.net/checkpoints/silver_valid_tweets"
            )
            .option("mergeSchema", "true")
            .toTable("silver_catalog1.processed.silver_valid_tweets")
    )

    silverQuery.awaitTermination()

    logger.info("Silver Valid Tweets Loaded Successfully")
    print("Silver Valid Tweets Loaded Successfully")

except Exception as e:

    logger.error(f"Silver Valid Tweets Pipeline Failed: {str(e)}")
    print(f"Silver Valid Tweets Pipeline Failed: {str(e)}")

    raise

Silver Valid Tweets Pipeline Started
Bronze Valid Tweets Table Read Successfully
Silver Valid Tweets Transformations Completed Successfully


Logger notebook executed successfully


In [0]:
%sql
SELECT COUNT(*) FROM silver_catalog1.processed.silver_valid_tweets;

count(1)
2604


In [0]:
%sql
SELECT * 
FROM silver_catalog1.processed.silver_valid_tweets
LIMIT 10;

tweet_id,topic_category,tweet_text,tweet_timestamp,impressions,likes,retweets,replies,engagement_count,sentiment_score,bronze_load_time,pipeline_name,source_system,ingestion_date,tweet_date,silver_load_time
T5446,AI,Streaming $$$ working,2025-01-16T22:50:00Z,8534,4995,706,0,888,0.608286357,2026-07-09T10:44:11.565Z,Silver_Valid_Tweets,Azure Event Hub,2026-07-09,2025-01-16,2026-07-10T04:47:34.565Z
T22443,SPORTS,Love this!!! 😊,2025-01-19T18:19:00Z,4457,3638,740,1739,946,0.239220852,2026-07-09T10:44:11.565Z,Silver_Valid_Tweets,Azure Event Hub,2026-07-09,2025-01-19,2026-07-10T04:47:34.565Z
T19981,AI,Love this!!! 😊,2025-01-22T22:49:00Z,18707,2871,615,1273,730,0.471464188,2026-07-09T10:44:11.565Z,Silver_Valid_Tweets,Azure Event Hub,2026-07-09,2025-01-22,2026-07-10T04:47:34.565Z
T34812,CLOUD,Error###Detected,2025-01-21T15:51:00Z,4224,476,83,1265,1695,0.246795072,2026-07-09T10:44:11.565Z,Silver_Valid_Tweets,Azure Event Hub,2026-07-09,2025-01-21,2026-07-10T04:47:34.565Z
T23458,FINANCE,Null?? value??,2025-01-07T22:17:00Z,17711,4034,498,1390,1636,0.498918531,2026-07-09T10:44:11.565Z,Silver_Valid_Tweets,Azure Event Hub,2026-07-09,2025-01-07,2026-07-10T04:47:34.565Z
T37939,SPORTS,Error###Detected,2025-01-14T11:04:00Z,10239,1734,144,0,770,0.30905506,2026-07-09T10:44:11.565Z,Silver_Valid_Tweets,Azure Event Hub,2026-07-09,2025-01-14,2026-07-10T04:47:34.565Z
T39272,CLOUD,Worst service ever :(,2025-01-05T02:09:00Z,2634,1480,183,604,640,0.745530222,2026-07-09T10:44:11.565Z,Silver_Valid_Tweets,Azure Event Hub,2026-07-09,2025-01-05,2026-07-10T04:47:34.565Z
T2796,SPORTS,Streaming $$$ working,2025-01-21T14:11:00Z,8581,1749,70,0,1467,0.101921028,2026-07-09T10:44:11.565Z,Silver_Valid_Tweets,Azure Event Hub,2026-07-09,2025-01-21,2026-07-10T04:47:34.565Z
T25759,FINANCE,Spark > Hadoop ???,2025-01-06T11:19:00Z,9734,2838,743,578,38,0.687462275,2026-07-09T10:44:11.565Z,Silver_Valid_Tweets,Azure Event Hub,2026-07-09,2025-01-06,2026-07-10T04:47:34.565Z
T29286,FINANCE,Love this!!! 😊,2025-01-05T08:24:00Z,9131,4613,362,112,801,0.557190366,2026-07-09T10:44:11.565Z,Silver_Valid_Tweets,Azure Event Hub,2026-07-09,2025-01-05,2026-07-10T04:47:34.565Z
